# Rust 115: Data Types and Their Libraries — Practice

Companion exercises for `15_data-types-and-their-libraries_e.ipynb`. Every section mirrors the section of the same number in the example notebook, and every demonstration cell has an exercise (§4.2+§4.3 and §6.1+§6.3 share theirs), so a `**See**` reference lands on the exact code that shows the technique.

The checks at the bottom of each cell are the grader. Edit only the parts marked `TODO`; when a cell runs without panicking you will see an `Exercise N.M passed` line.


### Contents

- [0. How to Complete These Exercises](#0-how-to-complete-these-exercises)
- [1. Time Is a Domain Problem](#1-time-is-a-domain-problem)
- [2. Two Ways to Say "Duration"](#2-two-ways-to-say-duration)
- [3. Chrono vs `time`: Two Crates, Same Problem](#3-chrono-vs-time-two-crates-same-problem)
- [4. Formatting and Parsing: The Boundary of Every System](#4-formatting-and-parsing-the-boundary-of-every-system)
- [5. Time Zones: The Same Instant, Different Walls](#5-time-zones-the-same-instant-different-walls)
- [6. Exact Money: `rust_decimal`](#6-exact-money-rust_decimal)
- [7. Identifiers: `uuid`](#7-identifiers-uuid)
- [8. The Row World and the Column World](#8-the-row-world-and-the-column-world)
- [9. Polars: Filter, Select, Sort](#9-polars-filter-select-sort)
- [10. Polars: `group_by` Aggregation](#10-polars-group_by-aggregation)
- [11. Tables Move as CSV: The Roundtrip](#11-tables-move-as-csv-the-roundtrip)
- [12. Choosing Representations Before Libraries](#12-choosing-representations-before-libraries)
- [13. Reading the Compiler: E-Codes](#13-reading-the-compiler-e-codes)
- [16. Review: Mixed Problems](#16-review-mixed-problems)

§1–§13 each have exercises for the demonstration cells of the example notebook (§14's receipts ledger and §15's cheat sheet are reading). When a check fails, the fastest route back is §15 of the example notebook, which is the cheat sheet.


## 0. How to Complete These Exercises

This notebook runs on the same **evcxr** kernel as the example notebook. Three consequences shape how you work:

1. **No `fn main`.** The body of a cell *is* the body of `main`. Type statements directly.
2. **State persists.** A binding from one cell is visible in every cell below it — including `use` lines.
3. **Items are order-independent.** A struct or impl may be used above the cell that defines it.

> **Item names must be unique across the whole notebook.** Every cell shares one namespace; two functions with the same name are a redefinition error even in different sections. The exercise names here are unique by construction — do not introduce colliding helpers. The same rule covers `use` lines: the first cell that needs a name does the import, and later cells rely on persistence — a repeated `use` is a redefinition (E0252).

### The stub convention

Each exercise cell declares a signature, a doc comment stating the contract, and a `// --- check (do not modify) ---` block of `assert_eq!`/`assert!` calls. The body you must fill in is `todo!()`, which compiles but **panics** when executed. Replace it and re-run; an `Exercise N.M passed` line means you are done.

### Four notes specific to this chapter

- **Imports arrive once, with the first cell that needs them.** §1.1 imports chrono's traits, §6.1 `rust_decimal`'s types, §9.1 the one polars glob — every later cell relies on persistence, exactly as the example notebook teaches in its §0.
- **Every timestamp in a check is fixed.** No `Utc::now()` anywhere — checks assert against `with_ymd_and_hms`-built instants, so they pass today and next year. `uuid` ids are checked by *structure* (length, version nibble) for the same reason.
- **Floats are never compared with `==`.** Where the row world's `f64` arithmetic is checked, the check asserts a tolerance — the example notebook's §6 is why the *exact* answers live in `Decimal`.
- **The crate cell below is given and must run first** (once per kernel session, features included). A `cannot find crate` diagnostic means it has not run — restart and go from the top.

### Expect warnings until you finish

Three warning families are normal and self-cleaning:

- `unused variable` — every unsolved stub whose parameters are untouched;
- `unused import` — the `use` lines serving a stub you have not solved yet;
- `never used`/`never read`/`never constructed` — stub-induced dead code: given structs and fields sit unread until your body reads them.

Each warning disappears when you implement the body that needs it. If a diagnostic appears that is not in one of those families, it is yours.

**See**: §0 of the example notebook — and its §13 when a diagnostic surprises you.


In [ ]:
// Given: the chapter's crates, one load per kernel session (run from the top
// after a restart). Features included — a re-`:dep` with different features is
// a rebuild, so change features only via restart.
:dep chrono = "0.4"
:dep time = { version = "0.3", features = ["macros", "formatting", "parsing"] }
:dep uuid = { version = "1", features = ["v4"] }
:dep rust_decimal = "1"
:dep polars = { version = "0.55", features = ["lazy", "csv", "temporal", "strings"] }


## 1. Time Is a Domain Problem

The calendar is a library's job, and the library validates it at construction: `from_ymd_opt` returns `Option` because "February 30" is not a date. Build instants through `Utc.with_ymd_and_hms`, read calendar facts through the `Datelike` trait (the trait import below is *the* chrono gotcha), and expect a parsed instant to equal a built one.

**See**: §1 of the example notebook.


In [ ]:
// Given imports: the traits every chrono field-read below needs (once).
use chrono::{Datelike, TimeZone, Utc};

/// Is (y, m, d) a real calendar date? The Option constructor already knows.
fn is_real_date(y: i32, m: u32, d: u32) -> bool {
    todo!()
}

/// Build the UTC instant for (y, m, d, h, mi, s) — the desk's settlement stamp.
fn settlement_utc(y: i32, m: u32, d: u32, h: u32, mi: u32, s: u32) -> chrono::DateTime<Utc> {
    todo!()
}

// --- check (do not modify) ---
assert!(!is_real_date(2026, 2, 30)); // February 30: never happened
assert!(!is_real_date(2026, 2, 29)); // 2026 is not a leap year
assert!(is_real_date(2028, 2, 29)); // 2028 is
let t = settlement_utc(2026, 9, 29, 12, 0, 0);
assert_eq!(t.weekday(), chrono::Weekday::Tue); // Datelike resolves weekday
assert_eq!((t.year(), t.month(), t.day()), (2026, 9, 29));
// a parsed instant equals a built one: a DateTime is a value, not a string
let from_wire = chrono::DateTime::parse_from_rfc3339("2026-09-29T12:00:00Z").unwrap();
assert_eq!(from_wire, t);
println!("Exercise 1.1 passed");


## 2. Two Ways to Say "Duration"

`std::time::Duration` is the unsigned machine span; `TimeDelta` is the signed calendar span whose arithmetic rolls dates. Subtract two instants and you get a `TimeDelta`; add a negative one and you travel back; `to_std()` is the honest bridge (it refuses negatives).

**See**: §2 of the example notebook.


In [ ]:
// Given: no new imports — TimeZone and Utc persist from §1.1.
use chrono::TimeDelta; // this section's one chrono import (§0's rule)

/// Seconds from `b` to `a` as an i64 (later-first: the close minus the open).
fn gap_seconds(a: chrono::DateTime<Utc>, b: chrono::DateTime<Utc>) -> i64 {
    todo!()
}

/// The instant `span` before `at` — a negative TimeDelta is a value, not an error.
fn back_up(at: chrono::DateTime<Utc>, span: chrono::TimeDelta) -> chrono::DateTime<Utc> {
    todo!()
}

// --- check (do not modify) ---
let open = settlement_utc(2026, 9, 29, 22, 0, 0); // 22:00, from §1.1
let close = settlement_utc(2026, 9, 30, 4, 0, 0); // 04:00 next day
assert_eq!(gap_seconds(close, open), 6 * 3600); // a six-hour session
let earlier = back_up(open, TimeDelta::hours(-2));
assert_eq!(earlier.format("%H:%M").to_string(), "20:00");
// calendar arithmetic: the day rolled under the addition
let day_older = open + TimeDelta::hours(30);
assert_eq!(day_older.format("%Y-%m-%d").to_string(), "2026-10-01");
// the bridge: negative spans have no std form
assert!(TimeDelta::hours(-2).to_std().is_err());
assert_eq!(TimeDelta::hours(2).to_std().unwrap(), std::time::Duration::from_secs(7200));
println!("Exercise 2.1 passed");


## 3. Chrono vs `time`: Two Crates, Same Problem

`time` decomposes the same domain: `Date` + `Time` → `PrimitiveDateTime`, months as a `Month` **enum** (13 cannot be written), `from_calendar_date` validated like chrono's `_opt` family, and `time::Duration` the signed span. Everything has a counterpart; the grammars differ.

**See**: §3 of the example notebook.


In [ ]:
// Given: no new imports — this section exercises the `time` crate by path.

/// Christmas Day 2026 as a `time` Date. The month is an ENUM: pick it, do not
/// spell a number.
fn christmas_2026() -> time::Date {
    todo!()
}

/// Glue date (from_calendar_date) and clock time into a PrimitiveDateTime.
/// `time::Time::from_hms` returns a Result - unwrap it.
fn at_open(d: time::Date) -> time::PrimitiveDateTime {
    todo!()
}

// --- check (do not modify) ---
let xmas = christmas_2026();
assert_eq!(xmas.year(), 2026);
assert_eq!(xmas.weekday(), time::Weekday::Friday);
// the enum month cannot hold 13 - the value itself does not exist:
// time::Month::December == 13 would not even compile
let session = at_open(xmas);
assert_eq!((session.hour(), session.minute()), (9, 30));
// both crates subtract a day the same way
assert_eq!((session - time::Duration::days(1)).day(), 24);
assert_eq!((settlement_utc(2026, 12, 25, 9, 30, 0) - chrono::TimeDelta::days(1)).day(), 24);
println!("Exercise 3.1 passed");


## 4. Formatting and Parsing: The Boundary of Every System

Formats are the wire: RFC 3339 at the edge (`2026-09-29T14:32:07Z`), custom templates inside. chrono's `%`-template is runtime data; `time`'s `format_description!` is checked at compile time. Both validate the calendar at the boundary — Sept 31 has the right digits and the wrong world.

**See**: §4 of the example notebook.


In [ ]:
// Given: no new imports - format_description! arrives by its full path here.

/// Format `at` as "YYYY-MM-DD HH:MM:SS" through chrono's runtime template.
fn chrono_stamp(at: chrono::DateTime<Utc>) -> String {
    todo!()
}

/// Roundtrip through the wire: to_rfc3339_opts then parse back. The result
/// must EQUAL the input instant.
fn via_wire(at: chrono::DateTime<Utc>) -> chrono::DateTime<chrono::FixedOffset> {
    todo!()
}

/// Compile-time template: build the description with time's macro, format with
/// it. Return Err when the text is not a valid date in that shape.
fn time_desc_format(d: time::Date) -> Result<String, time::error::Format> {
    todo!()
}

// --- check (do not modify) ---
let t = settlement_utc(2026, 9, 29, 14, 32, 7);
assert_eq!(chrono_stamp(t), "2026-09-29 14:32:07");
let back = via_wire(t);
assert_eq!(back.to_rfc3339_opts(chrono::SecondsFormat::Secs, true), "2026-09-29T14:32:07Z");
// the boundary validates the calendar, not just the shape:
assert!(chrono::DateTime::parse_from_rfc3339("2026-09-31T00:00:00Z").is_err());
let xmas = christmas_2026(); // §3.1
assert_eq!(
    time_desc_format(xmas).unwrap(),
    "2026-12-25"
);
// "yer" is a typo - but it is a COMPILE-time typo in a real template; here the
// runtime refusal is exercised on impossible calendar text instead:
assert!(time::Date::parse("2026-09-31", &time::macros::format_description!("[year]-[month]-[day]")).is_err());
println!("Exercise 4.1 passed");


## 5. Time Zones: The Same Instant, Different Walls

A zone is a lens, not a property of a moment: `with_timezone` re-wraps the *same* instant, so all views compare `==` while formatting differently. Fixed offsets are seconds; store and compute in UTC, convert at the display edge.

**See**: §5 of the example notebook.


In [ ]:
// Given: no new imports - FixedOffset comes by path, chrono traits persist.

/// The EST lens: a FixedOffset of -5 hours (seconds, not hours - east_opt takes
/// seconds and returns Option).
fn est() -> chrono::FixedOffset {
    todo!()
}

/// Re-lens `at` into `zone`. Equality with the original instant must hold.
fn local_view(at: chrono::DateTime<Utc>, zone: chrono::FixedOffset) -> chrono::DateTime<chrono::FixedOffset> {
    todo!()
}

// --- check (do not modify) ---
let t = settlement_utc(2026, 9, 29, 14, 0, 0);
let nyc = local_view(t, est());
assert_eq!(nyc.format("%H:%M").to_string(), "09:00"); // the wall moved
assert_eq!(nyc.format("%:z").to_string(), "-05:00"); // the offset rode along
assert_eq!(nyc, t); // ...but the instant did not
let tokyo_view = local_view(t, chrono::FixedOffset::east_opt(9 * 3600).unwrap());
assert_eq!(tokyo_view.format("%H:%M").to_string(), "23:00");
assert_eq!(nyc, tokyo_view); // equality is on instants, not walls
println!("Exercise 5.1 passed");


## 6. Exact Money: `rust_decimal`

`f64` cannot be money: `0.1 + 0.2` is `0.30000000000000004` in binary. `Decimal` is a mantissa-plus-scale value — exact through add/multiply (which *adds* scales), explicit at rounding, where the strategy is a named policy.

**See**: §6 of the example notebook.


In [ ]:
// Given imports: this section owns rust_decimal's (§0's rule).
use rust_decimal::Decimal;
use std::str::FromStr;

/// Parse a decimal price from wire text. Returns None when the text is not a
/// number at all (FromStr's error discarded).
fn parse_price(text: &str) -> Option<Decimal> {
    todo!()
}

/// price x shares, exactly: scales ADD (2dp price x integer count = 2dp result).
fn notional(price: Decimal, shares: i64) -> Decimal {
    todo!()
}

/// The desk's boundary rule: round to 2dp, half away from zero - the NAMED
/// strategy, not the banker's default.
fn settle(x: Decimal) -> Decimal {
    todo!()
}

// --- check (do not modify) ---
let px = parse_price("418.25").unwrap();
assert!(parse_price("not-a-price").is_none());
assert_eq!(px.to_string(), "418.25");
let n = notional(px, 200);
assert_eq!(n.to_string(), "83650.00"); // exact, 2dp - no float anywhere
// the float world, for contrast (tolerance, never ==):
let float_n = 418.25_f64 * 200.0;
assert!((float_n - 83_650.0).abs() < 1e-9);
assert_eq!(settle(parse_price("2.675").unwrap()).to_string(), "2.68"); // half-up policy
assert_eq!(settle(Decimal::from_str("1.005").unwrap()).to_string(), "1.01");
// value equality ignores scale; normalize() makes storage canonical
assert_eq!(Decimal::from_str("370.3500").unwrap(), Decimal::from_str("370.35").unwrap());
assert_eq!(Decimal::from_str("370.3500").unwrap().normalize().to_string(), "370.35");
println!("Exercise 6.1 passed");


In [ ]:
/// (6.2 continues §6) - the sum of a list of Decimal fills, exactly.
fn portfolio_value(fills: &[Decimal]) -> Decimal {
    todo!()
}

// --- check (do not modify) ---
let legs = vec![
    parse_price("0.10").unwrap(),
    parse_price("0.20").unwrap(),
    parse_price("0.10").unwrap(),
];
let total = portfolio_value(&legs);
assert_eq!(total, parse_price("0.40").unwrap()); // exact - floats owe an epsilon here
assert_eq!(total.to_string(), "0.40");
// 1/3 at full precision, trimmed EXPLICITLY:
let third = Decimal::from(1) / Decimal::from(3);
assert_eq!(third.to_string(), "0.3333333333333333333333333333");
assert_eq!(third.round_dp(4).to_string(), "0.3333");
println!("Exercise 6.2 passed");


## 7. Identifiers: `uuid`

Version 4 buys uniqueness with 122 random bits plus the spec's fingerprint nibbles: version at hex position 13 (`4`), variant at 17 (`8/9/a/b`). The type is a 16-byte `Copy` value; the string is only its display form.

**See**: §7 of the example notebook.


In [ ]:
/// A fresh order id: generate, then verify the SPEC, not the value - length 36,
/// version nibble '4' at hex[12], variant at hex[16] in {8,9,a,b}.
/// (Generation needs the v4 feature: the crate cell turned it on.)
fn fresh_order_id() -> String {
    let id = uuid::Uuid::new_v4();
    todo!()
}

/// Parse an id back to the value type; None on anything that is not a UUID.
fn scan_id(text: &str) -> Option<uuid::Uuid> {
    todo!()
}

// --- check (do not modify) ---
let s = fresh_order_id();
assert_eq!(s.len(), 36);
assert_eq!(s.as_bytes()[8], b'-');
let hex: String = s.chars().filter(|c| *c != '-').collect();
let nibbles: Vec<char> = hex.chars().collect();
assert_eq!(nibbles[12], '4'); // version 4, always
assert!(matches!(nibbles[16], '8' | '9' | 'a' | 'b')); // variant bits
let value = scan_id(&s).unwrap();
assert_eq!(value.to_string(), s); // the roundtrip is the identity
assert_eq!(value, uuid::Uuid::parse_str(&s).unwrap());
assert!(scan_id("not-a-uuid").is_none());
assert_eq!(uuid::Uuid::nil().to_string(), "00000000-0000-0000-0000-000000000000");
println!("Exercise 7.1 passed");


## 8. The Row World and the Column World

Structs at the boundary: a fill arrives whole, validates whole, lands whole. The one adapter the column world needs converts fills into polars `Row`s — whose `AnyValue::String` **borrows** the symbol, so the `Row<'a>` is tied to the input slice and is consumed immediately by `from_rows_and_schema`.

**See**: §8 of the example notebook.


In [ ]:
// Given: the record type and the adapter - both complete, both stay. Note the
// Row's borrowed lifetime and the fully-spelled paths (Row/AnyValue are NOT in
// the prelude).
struct Trade {
    symbol: String,
    price: f64,
    qty: i64,
}
fn blotter_rows(fills: &[Trade]) -> Vec<polars::frame::row::Row<'_>> {
    use polars::prelude::AnyValue;
    fills
        .iter()
        .map(|t| {
            polars::frame::row::Row(vec![
                AnyValue::String(&t.symbol),
                AnyValue::Float64(t.price),
                AnyValue::Int64(t.qty),
            ])
        })
        .collect()
}

/// Field-level validation, done the ROW-WORLD way: price positive, qty
/// nonzero. Return the first offending symbol, or None when the book is clean.
fn first_bad_fill(fills: &[Trade]) -> Option<&str> {
    todo!()
}

/// The order id for row `idx`, formatted - this is the DESK's view of a fill
/// (id, symbol, price x qty as "418.25x200"), not the column world's.
fn ticket_line(order_ids: &[uuid::Uuid], fills: &[Trade], idx: usize) -> String {
    todo!()
}

// --- check (do not modify) ---
let fills = vec![
    Trade { symbol: "AAPL".to_string(), price: 228.6, qty: 100 },
    Trade { symbol: "MSFT".to_string(), price: 418.0, qty: 200 },
    Trade { symbol: "NVDA".to_string(), price: 121.4, qty: 300 },
];
assert_eq!(first_bad_fill(&fills), None);
let dirty = vec![
    Trade { symbol: "GOOD".to_string(), price: 10.0, qty: 1 },
    Trade { symbol: "BAD".to_string(), price: -1.0, qty: 5 },
    Trade { symbol: "WORSE".to_string(), price: 5.0, qty: 0 },
];
assert_eq!(first_bad_fill(&dirty), Some("BAD")); // the FIRST offender wins
let ids: Vec<uuid::Uuid> = (0..3).map(|_| uuid::Uuid::new_v4()).collect();
let line = ticket_line(&ids, &fills, 0);
let parts: Vec<&str> = line.split_whitespace().collect();
assert_eq!(parts.len(), 3);
assert_eq!(parts[0], "AAPL");
assert_eq!(parts[1], scan_id(&parts[1]).unwrap().to_string()); // a real UUID
assert_eq!(parts[2], "228.6x100");
assert_eq!(blotter_rows(&fills).len(), 3); // the adapter is ready for §9
println!("Exercise 8.1 passed");


## 9. Polars: Filter, Select, Sort

The frame is built **with a declared schema** — `from_rows_and_schema` over §8's adapter, names and types stated, not inferred (positional rows alone would arrive as anonymous `column_1..`). Then the lazy grammar: filter rows, select columns, sort by name.

**See**: §9 of the example notebook.


In [ ]:
// Given: this cell owns the chapter's ONE polars glob import (§0's rule);
// every later cell relies on persistence.
use polars::prelude::*;

/// Build the blotter: §8's rows under the DECLARED schema (sym String,
/// px Float64, qty Int64).
fn build_blotter(fills: &[Trade]) -> polars::prelude::DataFrame {
    todo!()
}

/// Big ticks: rows with px > threshold, sym and px only, px descending.
fn big_ticks(blotter: &polars::prelude::DataFrame, threshold: f64) -> polars::prelude::DataFrame {
    todo!()
}

// --- check (do not modify) ---
let fills = vec![
    Trade { symbol: "AAPL".to_string(), price: 228.6, qty: 100 },
    Trade { symbol: "MSFT".to_string(), price: 418.0, qty: 200 },
    Trade { symbol: "AAPL".to_string(), price: 229.4, qty: 150 },
    Trade { symbol: "NVDA".to_string(), price: 121.4, qty: 300 },
];
let blotter = build_blotter(&fills);
assert_eq!(blotter.shape(), (4, 3));
assert_eq!(
    blotter.get_column_names().iter().map(|n| n.as_str()).collect::<Vec<_>>(),
    vec!["sym", "px", "qty"]
);
assert_eq!(blotter.dtypes(), vec![DataType::String, DataType::Float64, DataType::Int64]);
let out = big_ticks(&blotter, 200.0);
assert_eq!(out.shape(), (3, 2)); // MSFT, AAPL, AAPL - NVDA filtered out
let kept = out.column("sym").unwrap().str().unwrap();
assert_eq!(kept.get(0), Some("MSFT")); // descending by px
assert_eq!(kept.get(2), Some("AAPL"));
println!("Exercise 9.1 passed");


## 10. Polars: `group_by` Aggregation

The report is a plan: `group_by` splits by key, `agg` folds each group, `alias` names the computed column (the API-design move — a report should read like the desk talks). Sum keeps `Int64`; mean yields `Float64` — the operation picks the type.

**See**: §10 of the example notebook.


In [ ]:
// Given: no new imports - the glob persists from §9.1.

/// Per-symbol report: group by sym, total_qty = sum of qty, avg_px = mean of
/// px, sorted by sym. Chain one lazy plan; collect once.
fn per_symbol(blotter: &polars::prelude::DataFrame) -> polars::prelude::DataFrame {
    todo!()
}

// --- check (do not modify) ---
let fills = vec![
    Trade { symbol: "AAPL".to_string(), price: 228.6, qty: 100 },
    Trade { symbol: "MSFT".to_string(), price: 418.0, qty: 200 },
    Trade { symbol: "AAPL".to_string(), price: 229.4, qty: 150 },
    Trade { symbol: "NVDA".to_string(), price: 121.4, qty: 300 },
];
let report = per_symbol(&build_blotter(&fills));
assert_eq!(report.shape(), (3, 3)); // AAPL, MSFT, NVDA
let tq = report.column("total_qty").unwrap().i64().unwrap();
assert_eq!(tq.get(0), Some(250)); // AAPL: 100 + 150
assert_eq!(tq.get(1), Some(200));
assert_eq!(report.column("avg_px").unwrap().dtype(), &DataType::Float64); // mean's own type
let ap = report.column("avg_px").unwrap().f64().unwrap();
assert!((ap.get(0).unwrap() - 229.0).abs() < 1e-9); // (228.6 + 229.4) / 2
println!("Exercise 10.1 passed");


## 11. Tables Move as CSV: The Roundtrip

Tables cross boundaries as bytes: the schema-aware writer formats from column types, the reader **re-infers** the schema from cells — so write, read back, and *compare schemas*; the dtype equality is the contract that survived the trip. `Vec<u8>` + `Cursor` keeps it honest: no filesystem, pure boundary.

**See**: §11 of the example notebook.


In [ ]:
// Given: no new imports - CsvWriter, CsvReadOptions and Cursor are in scope
// through §9.1's glob (and std's Cursor through polars' re-export path used
// here in full).

/// Serialize the blotter to CSV bytes (header included).
fn to_csv(blotter: &polars::prelude::DataFrame) -> Vec<u8> {
    todo!()
}

/// Read bytes back into a frame - has_header(true), in-memory via Cursor.
fn from_csv(bytes: Vec<u8>) -> polars::prelude::DataFrame {
    todo!()
}

// --- check (do not modify) ---
let fills = vec![
    Trade { symbol: "AAPL".to_string(), price: 228.6, qty: 100 },
    Trade { symbol: "MSFT".to_string(), price: 418.0, qty: 200 },
];
let original = build_blotter(&fills);
let wire = to_csv(&original);
let text = String::from_utf8(wire.clone()).unwrap();
assert!(text.starts_with("sym,px,qty\n")); // header row: names, comma-joined
assert!(text.contains("AAPL,228.6,100"));
let back = from_csv(wire);
assert_eq!(back.shape(), (2, 3));
assert_eq!(
    back.get_column_names().iter().map(|n| n.as_str()).collect::<Vec<_>>(),
    vec!["sym", "px", "qty"]
);
assert_eq!(back.dtypes(), original.dtypes()); // the schema survived the trip
// the returned table is a REAL table - §10's report runs on it:
let again = per_symbol(&back);
assert_eq!(again.height(), 2);
println!("Exercise 11.1 passed");


## 12. Choosing Representations Before Libraries

The judgment cell: one `Fill` struct, five fields, five sections of the example chapter expressed as **field types** — `Uuid` for identity, `DateTime<Utc>` for the instant, `Decimal` for money, the record's own `String`/`i64` for the desk's fields. Choose the type first; the logic follows.

**See**: §12 of the example notebook.


In [ ]:
// Given: the correctly-represented record. Solve only the ledger functions.
struct Fill {
    id: uuid::Uuid,
    ts: chrono::DateTime<chrono::Utc>,
    symbol: String,
    price: rust_decimal::Decimal,
    qty: i64,
}
impl Fill {
    fn notional(&self) -> rust_decimal::Decimal {
        self.price * rust_decimal::Decimal::from(self.qty) // scales add: exact
    }
}

/// Every distinct symbol in the ledger, alphabetically. (Collect, sort, dedup
/// - or a BTreeSet; either is the row world doing right.)
fn symbols_in(ledger: &[Fill]) -> Vec<String> {
    todo!()
}

/// Total notional per symbol for ONE symbol, exactly (a Decimal sum - the
/// answer §6.2 already knows how to give).
fn symbol_notional(ledger: &[Fill], want: &str) -> rust_decimal::Decimal {
    todo!()
}

// --- check (do not modify) ---
let mk = |sym: &str, px: &str, qty: i64, minute: u32| Fill {
    id: uuid::Uuid::new_v4(),
    ts: settlement_utc(2026, 9, 29, 15, minute, 0),
    symbol: sym.to_string(),
    price: parse_price(px).unwrap(),
    qty,
};
let ledger = vec![mk("MSFT", "418.25", 200, 0), mk("AAPL", "228.60", 100, 1), mk("MSFT", "419.00", 50, 2)];
assert_eq!(symbols_in(&ledger), vec!["AAPL".to_string(), "MSFT".to_string()]);
let msft = symbol_notional(&ledger, "MSFT");
assert_eq!(msft, parse_price("104600.00").unwrap()); // 83650.00 + 20950.00, exact
assert_eq!(msft.to_string(), "104600.00");
assert_eq!(ledger[0].ts.format("%Y-%m-%d").to_string(), "2026-09-29");
assert_eq!(ledger[0].id.get_version_num(), 4);
println!("Exercise 12.1 passed");


## 13. Reading the Compiler: E-Codes

The catalogue's boundary rows: features gate APIs (E0432 until the right `:dep`), traits gate methods (E0599 until `Datelike`/`Timelike` are in scope), one namespace gates imports (E0252), and floats refuse silent `==` (E0308). One cell, every dodge.

**See**: §13 of the example notebook.


In [ ]:
/// (13.1 continues the catalogue) - the trait-visibility dodge, as a function:
/// return `at`'s hour. The method lives on Timelike - the import above the
/// stub block is given, your body just calls it.
use chrono::Timelike;
fn hour_of(at: chrono::DateTime<Utc>) -> u32 {
    todo!()
}

// --- check (do not modify) ---
// E0432 dodged: the features arrived with the crate cell (§0) - macros visible:
let desc = time::macros::format_description!("[year]-[month]-[day]");
assert_eq!(time::Date::from_ordinal_date(2026, 273).unwrap().format(&desc).unwrap(), "2026-09-30");
// E0599 dodged: Timelike is in scope (the given use line) - the method resolves:
assert_eq!(hour_of(settlement_utc(2026, 9, 29, 15, 4, 0)), 15);
// E0599 dodged, uuid row: the v4 feature is on, new_v4 exists:
assert_eq!(uuid::Uuid::new_v4().get_version_num(), 4);
// E0308 dodged: float equality asserted with tolerance, never bare ==:
assert!((0.1_f64 + 0.2_f64 - 0.3).abs() < 1e-9);
// ...or sidestepped in Decimal, which compares exactly:
assert_eq!(parse_price("0.1").unwrap() + parse_price("0.2").unwrap(), parse_price("0.3").unwrap());
// E0252 dodged: every use line in this notebook shipped exactly once.
println!("Exercise 13.1 passed");


## 16. Review: Mixed Problems

Three synthesis problems; each reaches across sections the way real work does. Same stub convention, same grader.

### 16.1 The desk's day, end to end

Fills in, tickets out: build the blotter, keep the big ticks, write them across the CSV boundary, read them back, and read one column of the returned table. Every stage of this chapter in one plan.

**See**: §8–§11 of the example notebook.


In [ ]:
// Given: no new items - every tool exists from §8 to §11. Compose them.

/// Count of rows in `frame` whose `sym` column equals `sym`.
fn count_symbol(frame: &polars::prelude::DataFrame, sym: &str) -> usize {
    todo!()
}

// --- check (do not modify) ---
let fills = vec![
    Trade { symbol: "AAPL".to_string(), price: 229.4, qty: 150 },
    Trade { symbol: "MSFT".to_string(), price: 418.0, qty: 200 },
    Trade { symbol: "NVDA".to_string(), price: 121.4, qty: 300 },
    Trade { symbol: "AAPL".to_string(), price: 428.6, qty: 100 },
];
let blotter = build_blotter(&fills);
let big = big_ticks(&blotter, 200.0); // §9.1: px > 200, sym+px, descending
let bytes = to_csv(&big); // §11.1: across the boundary
let back = from_csv(bytes);
assert_eq!(back.shape(), (3, 2));
// the read-back table answers real questions:
assert_eq!(count_symbol(&back, "AAPL"), 2); // 428.6 and 229.4 both survived
assert_eq!(count_symbol(&back, "NVDA"), 0); // filtered at §9, gone forever
let kept = back.column("sym").unwrap().str().unwrap();
assert_eq!(kept.get(0), Some("AAPL")); // 428.6 leads the descending frame
println!("Exercise 16.1 passed");


### 16.2 The settlement clock

A trade at 23:30 UTC, a desk that wants its wall clocks, and a report that must survive both: build the instant, re-lens it, walk the day boundary, and prove the wire format is the same instant.

**See**: §1, §2, §4, §5 of the example notebook.


In [ ]:
// Given: no new items - §1's builders, §2's spans, §5's lenses compose.

/// The UTC instant for (y, m, d, 23, 30, 0), viewed through `zone`, as
/// "YYYY-MM-DD HH:MM ±HH:MM" - the full wall stamp, offset included.
fn wall_stamp(y: i32, m: u32, d: u32, zone: chrono::FixedOffset) -> String {
    todo!()
}

// --- check (do not modify) ---
let trade = settlement_utc(2026, 9, 29, 23, 30, 0);
let est_zone = est(); // §5.1
let stamp = wall_stamp(2026, 9, 29, est_zone);
assert_eq!(stamp, "2026-09-29 18:30 -05:00"); // 23:30 UTC is 18:30 in New York
// the SAME instant through Tokyo's lens: next day's wall, same instant
let tokyo = wall_stamp(2026, 9, 29, chrono::FixedOffset::east_opt(9 * 3600).unwrap());
assert_eq!(tokyo, "2026-09-30 08:30 +09:00");
assert_eq!(
    chrono::DateTime::parse_from_rfc3339("2026-09-29T23:30:00Z").unwrap(),
    trade
); // the wire speaks the instant, not the wall
// and the calendar rolled under arithmetic, §2-style:
let next_day = trade + chrono::TimeDelta::hours(1);
assert_eq!(next_day.format("%Y-%m-%d").to_string(), "2026-09-30");
println!("Exercise 16.2 passed");


### 16.3 The audited ledger

The chapter's full stack on one problem: represent fills exactly, summarize them exactly, and stamp the summary with a well-formed id and an RFC 3339 close-of-day instant. Structure is the deliverable — the audit reads types, not strings.

**See**: §6, §7, §12 of the example notebook.


In [ ]:
// Given: the audit record's shape. The representation IS the answer.
struct Audit {
    report_id: uuid::Uuid,               // §7: 36 chars, version-nibbled
    closed_at: chrono::DateTime<chrono::Utc>, // §1: the instant, not a string
    total: rust_decimal::Decimal,        // §6: exact, not f64
    rows: usize,                         // the column world's count
}

/// Close the day: total notional across the ledger (one Decimal sum), the
/// ledger's length, a fresh report id, and the given closing instant.
fn close_day(ledger: &[Fill], closed_at: chrono::DateTime<chrono::Utc>) -> Audit {
    todo!()
}

// --- check (do not modify) ---
let ledger = vec![
    mk("MSFT", "418.25", 200, 0), // notional 83650.00
    mk("AAPL", "0.10", 100, 1),   // notional 10.00
    mk("NVDA", "121.45", 300, 2), // notional 36435.00
];
let close = settlement_utc(2026, 9, 29, 23, 59, 59);
let a = close_day(&ledger, close);
assert_eq!(a.total.to_string(), "120095.00"); // exact - float sums owe epsilon
assert_eq!(a.rows, 3);
let id_text = a.report_id.to_string();
assert_eq!(id_text.len(), 36);
let hex: String = id_text.chars().filter(|c| *c != '-').collect();
assert_eq!(hex.chars().nth(12).unwrap(), '4'); // v4's fingerprint
assert_eq!(a.closed_at, close);
assert_eq!(
    chrono::DateTime::parse_from_rfc3339(&a.closed_at.to_rfc3339_opts(chrono::SecondsFormat::Secs, true)).unwrap(),
    a.closed_at.with_timezone(&chrono::FixedOffset::east_opt(0).unwrap())
); // the audit's instant survives the wire
println!("Exercise 16.3 passed");
